In [1]:
import json
from pathlib import Path

# 1. Definición del Golden Set de 10 preguntas (conforme a la estructura del curso)
golden_set = [
    {
        "id": 1,
        "tipo": "simple",
        "pregunta": "How does Direct Preference Optimization (DPO) represent the optimal policy in terms of the ground-truth reward function?",
        "respuesta_esperada": "DPO uses an analytical mapping from the reward function to optimal policy, expressing the ground-truth reward implicitly via the log ratio between the policy and reference model.",
        "documentos_fuente": ["rafailov-2023-dpo.pdf"],
        "fragmento_esperado": "DPO directly optimizes for",
        "criterio_exito": "El sistema recupera un fragmento del documento fuente que contiene la frase y responde sin agregar información externa."
    },
    {
        "id": 2,
        "tipo": "simple",
        "pregunta": "How many parameters does the largest GPT-3 model architecture have according to Brown et al. (2020)?",
        "respuesta_esperada": "The largest GPT-3 model has 175 billion parameters.",
        "documentos_fuente": ["brown-2020-gpt3.pdf"],
        "fragmento_esperado": "175 billion parameters",
        "criterio_exito": "El sistema recupera el fragmento correcto."
    },
    {
        "id": 3,
        "tipo": "simple",
        "pregunta": "What are community summaries in the GraphRAG pipeline and what role do they play?",
        "respuesta_esperada": "Community summaries are hierarchical summaries generated by an LLM over graph communities to answer global sensemaking queries.",
        "documentos_fuente": ["edge-2024-graphrag.pdf"],
        "fragmento_esperado": "community summaries",
        "criterio_exito": "El sistema recupera el fragmento correcto."
    },
    {
        "id": 4,
        "tipo": "simple",
        "pregunta": "According to Ng and Jordan (2001), what is the sample complexity convergence rate for Naive Bayes compared to Logistic Regression?",
        "respuesta_esperada": "Naive Bayes converges to its asymptotic error with sample size O(log n), whereas Logistic Regression requires O(n) samples.",
        "documentos_fuente": ["ng-jordan-2001-discriminative-vs-generative.pdf"],
        "fragmento_esperado": "O(log n)",
        "criterio_exito": "El sistema recupera el fragmento correcto."
    },
    {
        "id": 5,
        "tipo": "multi-chunk",
        "pregunta": "Compare the training complexity and stability of DPO against standard PPO-based RLHF as described by Rafailov et al. (2023).",
        "respuesta_esperada": "PPO-based RLHF requires fitting an explicit reward model and training a policy via reinforcement learning with complex actor-critic updates, while DPO optimizes a single cross-entropy loss directly on preference pairs without reinforcement learning.",
        "documentos_fuente": ["rafailov-2023-dpo.pdf"],
        "fragmento_esperado": "",
        "criterio_exito": "El sistema recupera al menos un fragmento del documento fuente en el top-k."
    },
    {
        "id": 6,
        "tipo": "multi-chunk",
        "pregunta": "How do Naive Vector RAG and GraphRAG differ in handling high-level global sensemaking versus localized question answering?",
        "respuesta_esperada": "Naive RAG retrieves localized text chunks via cosine similarity and struggles with global dataset understanding, while GraphRAG constructs knowledge graphs and hierarchical community summaries to aggregate dataset-wide themes.",
        "documentos_fuente": ["edge-2024-graphrag.pdf"],
        "fragmento_esperado": "",
        "criterio_exito": "El sistema recupera fragmentos fuente relevantes."
    },
    {
        "id": 7,
        "tipo": "negativo",
        "pregunta": "What are the specific requirements for calculating corporate travel expense reimbursements at NimbusSoft?",
        "respuesta_esperada": "El corpus no contiene información suficiente.",
        "documentos_fuente": [],
        "fragmento_esperado": "",
        "criterio_exito": "No hay nada que recuperar: se mide la tasa de abstención correcta."
    },
    {
        "id": 8,
        "tipo": "negativo",
        "pregunta": "What protocol is mandated by the field practice guidelines regarding student accident insurance policies?",
        "respuesta_esperada": "El corpus no contiene información suficiente.",
        "documentos_fuente": [],
        "fragmento_esperado": "",
        "criterio_exito": "No hay nada que recuperar: se mide la tasa de abstención correcta."
    },
    {
        "id": 9,
        "tipo": "adversarial",
        "pregunta": "Why did the authors of the 2020 GPT-3 paper choose to implement a separate reward model based on DPO to train 175B parameters?",
        "respuesta_esperada": "El corpus no contiene información suficiente.",
        "documentos_fuente": [],
        "fragmento_esperado": "",
        "criterio_exito": "Pregunta adversarial sin fuente real en este contexto; se evalúa abstención."
    },
    {
        "id": 10,
        "tipo": "simple",
        "pregunta": "What is the primary architectural difference between bi-encoders and cross-encoders?",
        "respuesta_esperada": "Bi-encoders encode queries and documents independently, whereas cross-encoders process them jointly through self-attention for higher accuracy.",
        "documentos_fuente": ["edge-2024-graphrag.pdf"],
        "fragmento_esperado": "cross-encoder",
        "criterio_exito": "Recupera el fragmento fuente correctamente."
    }
]

# Guardar en archivo local JSON
ruta_gs = Path("golden_set.json")
ruta_gs.write_text(json.dumps(golden_set, indent=2, ensure_ascii=False))
print(f"✓ Golden set guardado exitosamente en '{ruta_gs}' con {len(golden_set)} preguntas.")

# 2. Funciones para métricas de recuperación (Hit Rate@k y MRR)
def hit_rate_k(rankings, relevantes, k):
    """Calcula el Hit Rate@k sobre las preguntas respondibles."""
    if not rankings:
        return 0.0
    hits = 0
    for rank, rel in zip(rankings, relevantes):
        top_k = set(rank[:k])
        if top_k.intersection(set(rel)):
            hits += 1
    return hits / len(rankings)

def mrr_k(rankings, relevantes, k):
    """Calcula el Mean Reciprocal Rank (MRR) con el mismo corte k."""
    if not rankings:
        return 0.0
    mrr_sum = 0.0
    for rank, rel in zip(rankings, relevantes):
        rr = 0.0
        for idx, doc in enumerate(rank[:k]):
            if doc in rel:
                rr = 1.0 / (idx + 1)
                break
        mrr_sum += rr
    return mrr_sum / len(rankings)

print("✓ Funciones de métricas (hit_rate_k, mrr_k) definidas correctamente.")


✓ Golden set guardado exitosamente en 'golden_set.json' con 10 preguntas.
✓ Funciones de métricas (hit_rate_k, mrr_k) definidas correctamente.


In [5]:
import json
from pathlib import Path
import numpy as np
from rag_pipeline import RagPipeline

# 1. Inicializar el pipeline RAG
pipeline = RagPipeline()

# 2. Cargar el Golden Set
ruta_gs = Path("golden_set.json")
if not ruta_gs.is_file():
    raise FileNotFoundError("No se encuentra 'golden_set.json'. Asegúrate de haberlo creado en la celda anterior.")

golden_set = json.loads(ruta_gs.read_text(encoding="utf-8"))
print(f"✓ Golden set cargado: {len(golden_set)} preguntas.")

# 3. Parámetros de evaluación
K_VALUES = [3, 5]
UMBRAL_ABSTENCION = 0.45  # Umbral base sobre el score coseno top-1

respondibles = []
negativas = []

for item in golden_set:
    q = item["pregunta"]
    tipo = item["tipo"]

    # Buscar fuentes con soporte para plural, singular o vacío
    docs_raw = item.get("documentos_fuente") or item.get("documento_fuente") or []
    if isinstance(docs_raw, str):
        docs_raw = [docs_raw]
    docs_esperados = set(docs_raw)

    # Recuperar top-k máximo (k=5) de Qdrant utilizando el pipeline
    hits = pipeline.retrieve(q, top_k=max(K_VALUES))

    # Obtener el score del mejor hit (top-1) para la abstención
    top_score = hits[0]["score"] if hits else 0.0

    # Lista de documentos recuperados en el top-k
    docs_recuperados = [h.get("document", "") for h in hits]

    eval_item = {
        "id": item.get("id"),
        "tipo": tipo,
        "pregunta": q,
        "top_score": top_score,
        "hits": hits,
        "docs_recuperados": docs_recuperados,
        "documentos_fuente": list(docs_esperados)
    }

    if tipo in ["simple", "multi-chunk", "adversarial"] and docs_esperados:
        eval_item["es_respondible"] = True
        respondibles.append(eval_item)
    else:
        eval_item["es_respondible"] = False
        negativas.append(eval_item)

print(f"✓ Evaluadas {len(respondibles)} preguntas respondibles y {len(negativas)} preguntas negativas/adversariales.")

# 4. Cálculo de Hit Rate@k y MRR sobre respondibles
print("\n" + "="*50)
print("RESULTADOS DE RECUPERACIÓN (Respondibles)")
print("="*50)

metricas_k = {}
for k in K_VALUES:
    hits_count = 0
    mrr_sum = 0.0

    for item in respondibles:
        docs_esp = set(item["documentos_fuente"])
        recup_k = item["docs_recuperados"][:k]

        # Hit Rate@k
        acertio = any(doc in docs_esp for doc in recup_k)
        if acertio:
            hits_count += 1

        # MRR@k
        rr = 0.0
        for idx, doc in enumerate(recup_k):
            if doc in docs_esp:
                rr = 1.0 / (idx + 1)
                break
        mrr_sum += rr

    hr = hits_count / len(respondibles) if respondibles else 0.0
    mrr = mrr_sum / len(respondibles) if respondibles else 0.0

    metricas_k[k] = {"Hit Rate": hr, "MRR": mrr}
    print(f"Hit Rate@{k}: {hr:.4f} | MRR@{k}: {mrr:.4f}")

# 5. Cálculo de las dos Tasas de Abstención
abstuvo_correctamente = 0
for item in negativas:
    if item["top_score"] < UMBRAL_ABSTENCION:
        abstuvo_correctamente += 1

tasa_abstencion_correcta = abstuvo_correctamente / len(negativas) if negativas else 0.0

abstuvo_indebidamente = 0
for item in respondibles:
    if item["top_score"] < UMBRAL_ABSTENCION:
        abstuvo_indebidamente += 1

tasa_abstencion_indebida = abstuvo_indebidamente / len(respondibles) if respondibles else 0.0

print("\n" + "="*50)
print("TASAS DE ABSTENCIÓN")
print("="*50)
print(f"Tasa de Abstención Correcta (en negativas):  {tasa_abstencion_correcta:.4f}")
print(f"Tasa de Abstención Indebida (en respondibles): {tasa_abstencion_indebida:.4f}")


✓ Golden set cargado: 10 preguntas.
✓ Evaluadas 7 preguntas respondibles y 3 preguntas negativas/adversariales.

RESULTADOS DE RECUPERACIÓN (Respondibles)
Hit Rate@3: 1.0000 | MRR@3: 1.0000
Hit Rate@5: 1.0000 | MRR@5: 1.0000

TASAS DE ABSTENCIÓN
Tasa de Abstención Correcta (en negativas):  0.3333
Tasa de Abstención Indebida (en respondibles): 0.0000
